## rMATS significant genes ∩ DEGs (up/down) overlap

Purpose:
- Read rMATS significant genes (`gene_name`, `gene_id`, `event_types`) from `rmats_significant_unique_genes_with_id_and_types.tsv`.
- Read DE gene lists (up and down) from DESeq2 outputs.
- Find intersection by either gene_name or gene_id, and annotate event types and DE status (up/down/both).
- Save annotated overlap to TSV in the rMATS summary output directory.

Steps: set paths below, then run all cells.


In [4]:
# ========== Parameters ==========
from pathlib import Path

# Input files
RMATS_AGG = "/path/to/rmat/output/rmats_summary/rmats_significant_unique_genes_with_id_and_types.tsv"
DEG_UP = "/path/to/rnaseq/deg/deseq2_up.txt"
DEG_DOWN = "/path/to/rnaseq/deg/deseq2_down.txt"

# Output directory
OUTPUT_DIR = "/path/to/rmat/output/overlapDEG"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print("Parameters set.")


Parameters set.


In [5]:
# ========== Utilities ==========
import re
import pandas as pd
import numpy as np
from pathlib import Path


def _first_clean_token(s: str) -> str:
    if pd.isna(s):
        return ''
    # take first token by comma/semicolon, trim
    return str(s).split(',')[0].split(';')[0].strip()


def load_rmats_agg(fp: str) -> pd.DataFrame:
    p = Path(fp)
    if not p.exists():
        raise FileNotFoundError(f"rMATS aggregated file not found: {fp}")
    df = pd.read_csv(p, sep='\t', dtype=str)
    # normalize expected columns
    # expect: gene_name, gene_id, event_types
    colmap = {c.lower(): c for c in df.columns}
    need = ['gene_name', 'gene_id', 'event_types']
    for c in need:
        if c not in colmap:
            raise ValueError(f"Missing column '{c}' in {fp}")
    out = pd.DataFrame({
        'gene_name': df[colmap['gene_name']].map(_first_clean_token).fillna('').astype(str).str.strip(),
        'gene_id': df[colmap['gene_id']].map(_first_clean_token).fillna('').astype(str).str.strip(),
        'event_types': df[colmap['event_types']].fillna('').astype(str),
    })
    return out


def _infer_gene_column(cols):
    # heuristic for DEG files
    priority = [
        'gene', 'gene_name', 'symbol', 'genes', 'id', 'geneid', 'gene_id', 'ensembl',
        'genename', 'gene.symbol', 'gene.symbols'
    ]
    low = {c.lower(): c for c in cols}
    for k in priority:
        if k in low:
            return low[k]
    # default to first column
    return cols[0]


def read_deg_gene_set(fp: str) -> set:
    p = Path(fp)
    if not p.exists():
        raise FileNotFoundError(f"DE file not found: {fp}")
    # Try TSV first, fallback to whitespace
    try:
        df = pd.read_csv(p, sep='\t', dtype=str, engine='python')
    except Exception:
        df = pd.read_csv(p, sep='\s+', dtype=str, engine='python', header=0)
    if df.shape[1] == 0:
        return set()
    gene_col = _infer_gene_column(list(df.columns))
    vals = df[gene_col].dropna().map(_first_clean_token).astype(str).str.strip()
    # also consider second common column if present and looks like id/symbol
    genes = {v for v in vals if v}
    return genes


print("Utilities loaded.")


Utilities loaded.


In [ ]:
# ========== Load, intersect, annotate, export ==========
import pandas as pd
from pathlib import Path

# 1) Load inputs
rmats = load_rmats_agg(RMATS_AGG)
up_set = read_deg_gene_set(DEG_UP)
down_set = read_deg_gene_set(DEG_DOWN)

print(f"rMATS genes (rows): {len(rmats)}  | unique gene_names: {rmats['gene_name'].ne('').sum()}  | unique gene_ids: {rmats['gene_id'].ne('').sum()}")
print(f"DEG up: {len(up_set)} | DEG down: {len(down_set)}")

# 2) Determine overlap (by gene_name OR gene_id)
rmats_nonempty = rmats.copy()
rmats_nonempty['gene_name'] = rmats_nonempty['gene_name'].fillna('').astype(str).str.strip()
rmats_nonempty['gene_id'] = rmats_nonempty['gene_id'].fillna('').astype(str).str.strip()

up_hit = rmats_nonempty['gene_name'].isin(up_set) | rmats_nonempty['gene_id'].isin(up_set)
down_hit = rmats_nonempty['gene_name'].isin(down_set) | rmats_nonempty['gene_id'].isin(down_set)

mask = up_hit | down_hit
overlap = rmats_nonempty.loc[mask].copy()

# 3) Annotate DE status
status = []
for u, d in zip(up_hit[mask].tolist(), down_hit[mask].tolist()):
    if u and d:
        status.append('both')
    elif u:
        status.append('up')
    elif d:
        status.append('down')
    else:
        status.append('')
overlap['DE_status'] = status

# 4) Sort and export
overlap = overlap[['gene_name', 'gene_id', 'event_types', 'DE_status']]
overlap = overlap.sort_values(['DE_status', 'gene_name', 'gene_id'], ascending=[True, True, True]).reset_index(drop=True)

out_path = Path(OUTPUT_DIR) / 'rmats_deg_overlap.tsv'
overlap.to_csv(out_path, sep='\t', index=False)

# 5) Print brief summary
num_total = len(overlap)
num_up = int((overlap['DE_status'] == 'up').sum())
num_down = int((overlap['DE_status'] == 'down').sum())
num_both = int((overlap['DE_status'] == 'both').sum())

print(f"Overlap genes: {num_total} (up={num_up}, down={num_down}, both={num_both})")
print('Saved:', out_path)
print('\nPreview:')
print(overlap.head(20).to_string(index=False))


rMATS genes (rows): 797  | non-empty gene_names: 720  | non-empty gene_ids: 797
DEG up raw: 1068 | DEG down raw: 789
DEG up tokens: 1679 | DEG down tokens: 1043
Matches via name → up=0, down=0; via ID → up=0, down=0
Overlap genes: 0 (up=0, down=0, both=0)
Saved: /path/to/rmat/output/overlapDEG/rmats_deg_overlap.tsv

Preview:
Empty DataFrame
Columns: [gene_name, gene_id, event_types, DE_status]
Index: []
